In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ====== CONFIG ======
PICKED_IPS_JSON = "/content/drive/MyDrive/Thesis/picked_ips.json"
TARGET_COL = "n_bytes"
MIN_POINTS = 50

TRAIN_RATIO = 0.35
VAL_RATIO   = 0.05

PLOT_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/ETS_evaluation/SES_Plots_mean_style")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ====== LOAD JSON ======
with open(PICKED_IPS_JSON, "r") as f:
    picked_files = json.load(f)

print(f"Loaded {len(picked_files)} processed IP CSV paths.")

for ip_path in picked_files:
    ip_path = Path(ip_path)

    # --- Load data ---
    try:
        df = pd.read_csv(ip_path)
    except Exception as e:
        print(f"[SKIP] Could not read {ip_path}: {e}")
        continue

    if "time" not in df.columns:
        print(f"[SKIP] No 'time' column in {ip_path}")
        continue

    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df = df.dropna(subset=["time"]).sort_values("time")

    y = df[TARGET_COL].fillna(0).values
    t = df["time"].values

    N = len(y)
    if N < MIN_POINTS:
        print(f"[SKIP] Too few points in {ip_path}")
        continue

    #Splt
    train_end = int(TRAIN_RATIO * N)
    val_end   = int((TRAIN_RATIO + VAL_RATIO) * N)

    y_train = y[:train_end]
    y_test  = y[val_end:]

    if len(y_test) == 0:
        print(f"[SKIP] No test set for {ip_path}")
        continue

    # --- Fit SES on TRAIN ---
    try:
        ses_model = ExponentialSmoothing(
            y_train,
            trend=None,
            seasonal=None
        ).fit(optimized=True)
    except Exception as e:
        print(f"[SKIP] SES failed for {ip_path}: {e}")
        continue

    # Forecast for the test horizon
    y_pred_test = ses_model.forecast(len(y_test))

    # Build full-length prediction array (NaN outside test)
    y_pred_full = np.full_like(y, np.nan, dtype=float)
    y_pred_full[val_end:] = y_pred_test

    # ===== PLOT: identical style to mean model =====
    plt.figure(figsize=(16, 4))
    plt.plot(t, y, label="Actual (n_bytes)")
    plt.plot(t, y_pred_full, "--", label="SES Prediction")

    plt.title(f"SES Model Forecast — {ip_path.name}")
    plt.xlabel("Time")
    plt.ylabel("Bytes")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    out_path = PLOT_DIR / f"{ip_path.stem}_ses_forecast_mean_style.png"
    plt.savefig(out_path, dpi=150)
    plt.close()

    print(f"[PLOT] Saved SES mean-style forecast for {ip_path.name} -> {out_path}")
